# ASMR: Scaling Analysis

This notebook analyzes ASMR performance characteristics at scale.

In [ ]:
import sys
sys.path.insert(0, '..')

import time
from datetime import datetime, timedelta
import random

## Latency Components

In [ ]:
print("ASMR Latency Components")
print("="*50)
print()
print("1. Embedding Generation: ~10-50ms")
print("   - Depends on embedding model")
print("   - sentence-transformers: ~20ms")
print("   - OpenAI API: ~100-200ms")
print()
print("2. Candidate Generation (FAISS/ChromaDB): ~1-10ms")
print("   - O(log n) for approximate search")
print("   - Sub-millisecond for <100K memories")
print()
print("3. Agent Reasoning (LLM calls): ~500-2000ms")
print("   - Dominates total latency")
print("   - Can be parallelized (relevance + recency)")
print("   - 4 agents × ~500ms = ~2s sequential")
print("   - With parallelization: ~1-1.5s")
print()
print("4. Synthesis: ~500-1000ms")
print("   - Single LLM call")
print("   - Depends on context size")

## Memory Scaling

In [ ]:
# Simulated scaling analysis
memory_counts = [100, 1000, 10000, 100000, 1000000]

print("Memory Store Scaling (FAISS IndexFlatIP)")
print("="*50)
print()
print("| Memories | Index Size | Search Time | Memory Usage |")
print("|----------|------------|-------------|--------------|")

for n in memory_counts:
    # 384-dim float32 embeddings
    size_mb = (n * 384 * 4) / (1024 * 1024)
    # Flat index: O(n) for exact, but very fast with SIMD
    search_time = 0.001 * (n / 1000)  # ~1ms per 1000 memories
    search_time = max(0.1, min(search_time, 100))  # Bound
    
    print(f"| {n:>8,} | {size_mb:>8.1f} MB | {search_time:>9.1f} ms | {size_mb * 1.5:>10.1f} MB |")

## Cost Analysis

In [ ]:
print("Cost per Query (GPT-4o-mini pricing)")
print("="*50)
print()
print("Assumptions:")
print("  - 50 candidate memories")
print("  - ~200 tokens per memory")
print("  - 4 agent calls")
print("  - GPT-4o-mini: $0.15/1M input, $0.60/1M output")
print()

candidates = 50
tokens_per_memory = 200
prompt_overhead = 500  # System prompt per agent
output_tokens = 500  # Agent response

# Per agent
input_tokens = (candidates * tokens_per_memory) + prompt_overhead
cost_input = (input_tokens / 1_000_000) * 0.15
cost_output = (output_tokens / 1_000_000) * 0.60
cost_per_agent = cost_input + cost_output

# Total (4 agents, but synthesis has smaller input)
total_cost = cost_per_agent * 4  # Simplified

print(f"Per agent call:")
print(f"  Input tokens: ~{input_tokens:,}")
print(f"  Output tokens: ~{output_tokens}")
print(f"  Cost: ${cost_per_agent:.5f}")
print()
print(f"Total per query: ${total_cost:.5f}")
print(f"Cost per 1000 queries: ${total_cost * 1000:.2f}")

## Optimization Strategies

In [ ]:
print("ASMR Optimization Strategies")
print("="*50)
print()
print("1. FAST MODE (skip agents)")
print("   - Use for low-stakes queries")
print("   - Latency: ~50ms")
print("   - Cost: $0 (no LLM calls)")
print()
print("2. PARALLEL AGENTS")
print("   - Run relevance + recency concurrently")
print("   - Reduces latency by ~30%")
print()
print("3. REDUCE CANDIDATES")
print("   - Lower top_k for candidate generation")
print("   - 50 → 20 candidates = ~60% token reduction")
print()
print("4. USE SMALLER MODELS")
print("   - GPT-4o-mini for most agents")
print("   - GPT-4o only for synthesis (if needed)")
print()
print("5. BATCH PROCESSING")
print("   - For bulk operations")
print("   - Amortize embedding costs")
print()
print("6. CACHING")
print("   - Cache query embeddings")
print("   - Cache common agent decisions")

## When to Use ASMR vs Simple RAG

In [ ]:
print("Decision Matrix: ASMR vs Simple RAG")
print("="*50)
print()
print("USE ASMR WHEN:")
print("  ✓ Information changes over time (policies, prices, personnel)")
print("  ✓ Contradictions are common in your data")
print("  ✓ Accuracy is more important than latency")
print("  ✓ False positives from embedding similarity are problematic")
print("  ✓ You need audit trails for retrieval decisions")
print()
print("USE SIMPLE RAG WHEN:")
print("  ✓ Data is static (documentation, fixed reference)")
print("  ✓ Sub-100ms latency required")
print("  ✓ Cost is a primary concern")
print("  ✓ Simple keyword/topic matching suffices")
print()
print("HYBRID APPROACH:")
print("  ✓ Use ASMR fast mode for initial retrieval")
print("  ✓ Trigger full reasoning when confidence is low")
print("  ✓ Cache ASMR results for common queries")